In [2]:
## 1. Prepare the DocShield Repository

import os
from pathlib import Path

REPO_DIR = Path("/content/docshield-ai")

if REPO_DIR.exists():
    %cd /content/docshield-ai
    !git pull
else:
    %cd /content
    !git clone https://github.com/Raghavtripathii/docshield-ai.git
    %cd /content/docshield-ai

print("Repository:", Path.cwd())

assert Path("src").exists()
assert Path("configs").exists()

print("Repository preparation: PASSED")

/content
Cloning into 'docshield-ai'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 127 (delta 55), reused 87 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 763.48 KiB | 3.88 MiB/s, done.
Resolving deltas: 100% (55/55), done.
/content/docshield-ai
Repository: /content/docshield-ai
Repository preparation: PASSED


In [3]:
## 2. Install the Reproducible Project Dependencies

!python -m pip install -q -r requirements-dev.txt

print("Dependency installation: COMPLETE")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 913.3/913.3 kB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 5.3 MB/s eta 0:00:00
Dependency installation: COMPLETE


In [4]:
## 3. Validate the Evaluation Environment

import sys
import torch
import transformers
import numpy as np

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), "GPU runtime is required."

print("GPU:", torch.cuda.get_device_name(0))

print("Evaluation environment: PASSED")

Python: 3.12.13
PyTorch: 2.11.0+cu128
Transformers: 5.13.1
CUDA available: True
GPU: Tesla T4
Evaluation environment: PASSED


In [5]:
## 4. Load the Versioned Baseline Configuration

import json
from pathlib import Path

CONFIG_PATH = Path("configs/baseline.json")

with CONFIG_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    baseline_config = json.load(file)

print(
    json.dumps(
        baseline_config,
        indent=2,
    )
)

assert (
    baseline_config["model_name"]
    == "microsoft/layoutlmv3-base"
)

print("Baseline configuration: PASSED")

{
  "experiment_name": "layoutlmv3_funsd_baseline",
  "model_name": "microsoft/layoutlmv3-base",
  "dataset_name": "nielsr/funsd",
  "seed": 42,
  "max_length": 512,
  "learning_rate": 5e-05,
  "train_batch_size": 2,
  "eval_batch_size": 2,
  "gradient_accumulation_steps": 2,
  "num_train_epochs": 3,
  "weight_decay": 0.01,
  "warmup_ratio": 0.1,
  "fp16": true,
  "save_strategy": "epoch",
  "logging_steps": 10
}
Baseline configuration: PASSED


In [8]:
## 5. Restore the Fine-Tuned LayoutLMv3 Checkpoint

from pathlib import Path
import zipfile
import shutil

ZIP_PATH = Path("/content/docshield-layoutlmv3-baseline.zip")
MODEL_DIR = Path("/content/docshield-layoutlmv3-baseline")

assert ZIP_PATH.exists(), "Checkpoint ZIP is not available."

# Start with a clean model directory.
if MODEL_DIR.exists():
    shutil.rmtree(MODEL_DIR)

MODEL_DIR.mkdir(parents=True, exist_ok=True)

# The checkpoint ZIP stores model files directly at its root,
# so extract them explicitly into MODEL_DIR.
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    archive.extractall(MODEL_DIR)

required_files = {
    "config.json",
    "model.safetensors",
    "processor_config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "training_args.bin",
}

restored_files = {
    path.name
    for path in MODEL_DIR.iterdir()
    if path.is_file()
}

missing_files = required_files - restored_files

assert not missing_files, (
    f"Checkpoint is incomplete. Missing: {sorted(missing_files)}"
)

print("Fine-tuned checkpoint: RESTORED")
print("Checkpoint path:", MODEL_DIR)

print("\nCheckpoint files:")
for file in sorted(restored_files):
    print(" -", file)

print("\nCheckpoint validation: PASSED")

Fine-tuned checkpoint: RESTORED
Checkpoint path: /content/docshield-layoutlmv3-baseline

Checkpoint files:
 - config.json
 - model.safetensors
 - processor_config.json
 - tokenizer.json
 - tokenizer_config.json
 - training_args.bin

Checkpoint validation: PASSED


In [7]:
## 5A. Inspect the Restored Checkpoint Structure

from pathlib import Path
import zipfile

zip_path = Path("/content/docshield-layoutlmv3-baseline.zip")

assert zip_path.exists(), "Checkpoint ZIP is missing."

with zipfile.ZipFile(zip_path, "r") as archive:
    members = archive.namelist()

print("Files stored inside ZIP:", len(members))

print("\nFirst 30 ZIP entries:")
for name in members[:30]:
    print(" -", name)

print("\nDirectories currently under /content:")
for path in sorted(Path("/content").iterdir()):
    print(
        " -",
        path.name,
        "[DIR]" if path.is_dir() else "[FILE]"
    )

Files stored inside ZIP: 6

First 30 ZIP entries:
 - processor_config.json
 - tokenizer.json
 - model.safetensors
 - tokenizer_config.json
 - config.json
 - training_args.bin

Directories currently under /content:
 - .config [DIR]
 - config.json [FILE]
 - docshield-ai [DIR]
 - docshield-layoutlmv3-baseline.zip [FILE]
 - model.safetensors [FILE]
 - processor_config.json [FILE]
 - sample_data [DIR]
 - tokenizer.json [FILE]
 - tokenizer_config.json [FILE]
 - training_args.bin [FILE]


In [9]:
## 6. Load and Validate the FUNSD Dataset

from src.data import (
    load_funsd,
    validate_columns,
    dataset_summary,
)

dataset = load_funsd()

validate_columns(dataset)

summary = dataset_summary(dataset)

print(dataset)

print("\nDataset sizes:")

for split, size in summary.items():
    print(f"{split}: {size}")

assert len(dataset["train"]) == 149
assert len(dataset["test"]) == 50

print("\nFUNSD dataset: PASSED")

README.md:   0%|          | 0.00/755 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 12.3MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.38MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/149 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'words', 'bboxes', 'ner_tags', 'image'],
        num_rows: 149
    })
    test: Dataset({
        features: ['id', 'words', 'bboxes', 'ner_tags', 'image'],
        num_rows: 50
    })
})

Dataset sizes:
train: 149
test: 50

FUNSD dataset: PASSED


In [10]:
## 7. Recover the FUNSD Label Vocabulary

from src.labels import get_label_list

label_list = get_label_list(dataset)

label2id = {
    label: index
    for index, label in enumerate(label_list)
}

id2label = {
    index: label
    for index, label in enumerate(label_list)
}

print("Number of labels:", len(label_list))

for index, label in enumerate(label_list):
    print(index, "->", label)

assert len(label_list) > 0

print("\nLabel vocabulary: PASSED")

Number of labels: 7
0 -> O
1 -> B-HEADER
2 -> I-HEADER
3 -> B-QUESTION
4 -> I-QUESTION
5 -> B-ANSWER
6 -> I-ANSWER

Label vocabulary: PASSED


In [11]:
## 8. Load the Fine-Tuned LayoutLMv3 Model

from transformers import (
    AutoProcessor,
    AutoModelForTokenClassification,
)

processor = AutoProcessor.from_pretrained(
    MODEL_DIR,
    apply_ocr=False,
)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_DIR,
)

model = model.to("cuda")
model.eval()

print(
    "Model:",
    model.__class__.__name__,
)

print(
    "Model labels:",
    model.config.num_labels,
)

assert model.config.num_labels == len(label_list)

print("Fine-tuned model loading: PASSED")

Loading weights:   0%|          | 0/214 [00:00<?, ?it/s]

Model: LayoutLMv3ForTokenClassification
Model labels: 7
Fine-tuned model loading: PASSED


In [13]:
## Diagnostic — Inspect the Actual Preprocessing API

import src.preprocessing as preprocessing

print("preprocessing.py location:")
print(preprocessing.__file__)

print("\nAvailable public names:")
for name in sorted(dir(preprocessing)):
    if not name.startswith("_"):
        print(" -", name)

preprocessing.py location:
/content/docshield-ai/src/preprocessing.py

Available public names:
 - Any
 - CONFIG
 - encode_document
 - validate_encoding


In [15]:
## Diagnostic — Find the Existing FUNSD Dataset Variable

from datasets import Dataset, DatasetDict

print("Dataset variables currently available:\n")

found = False

for name, value in list(globals().items()):
    if isinstance(value, (Dataset, DatasetDict)):
        found = True
        print(f"{name}: {type(value).__name__}")

        if isinstance(value, DatasetDict):
            print("  splits:", list(value.keys()))
            for split in value:
                print(f"  {split}: {len(value[split])}")
        else:
            print("  rows:", len(value))

        print()

if not found:
    print("NO DATASET VARIABLES FOUND")

Dataset variables currently available:

dataset: DatasetDict
  splits: ['train', 'test']
  train: 149
  test: 50



In [16]:
## Diagnostic — Inspect encode_document Signature

import inspect
from src.preprocessing import encode_document, validate_encoding

print("encode_document signature:")
print(inspect.signature(encode_document))

print("\nvalidate_encoding signature:")
print(inspect.signature(validate_encoding))

encode_document signature:
(processor: Any, image: Any, words: list[str], boxes: list[list[int]], labels: list[int])

validate_encoding signature:
(encoding: Any) -> None


In [21]:
## 9. Encode the FUNSD Evaluation Split

from src.preprocessing import (
    encode_document,
    validate_encoding,
)

def encode_example(example):
    encoding = encode_document(
        processor=processor,
        image=example["image"],
        words=example["words"],
        boxes=example["bboxes"],
        labels=example["ner_tags"],
    )

    validate_encoding(encoding)

    # encode_document returns tensors with a batch dimension of 1.
    # Dataset rows must represent individual unbatched samples.
    encoding = {
        key: value.squeeze(0)
        for key, value in encoding.items()
    }

    return encoding


eval_dataset = dataset["test"].map(
    encode_example,
    remove_columns=dataset["test"].column_names,
)

required_columns = {
    "input_ids",
    "attention_mask",
    "bbox",
    "pixel_values",
    "labels",
}

assert len(eval_dataset) == 50

assert required_columns.issubset(
    set(eval_dataset.column_names)
)

print("Evaluation rows:", len(eval_dataset))
print("Encoded columns:", eval_dataset.column_names)

print("\nEvaluation encoding: PASSED")

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Evaluation rows: 50
Encoded columns: ['input_ids', 'attention_mask', 'bbox', 'labels', 'pixel_values']

Evaluation encoding: PASSED


In [23]:
## 10. Construct the Baseline Evaluation Engine

from transformers import (
    Trainer,
    TrainingArguments,
    default_data_collator,
)

evaluation_args = TrainingArguments(
    output_dir="/content/docshield-evaluation",
    per_device_eval_batch_size=1,
    fp16=True,
    eval_accumulation_steps=1,
    report_to="none",
    remove_unused_columns=False,
)

evaluator = Trainer(
    model=model,
    args=evaluation_args,
    eval_dataset=eval_dataset,
    data_collator=default_data_collator,
    processing_class=processor,
)

print(
    "Evaluation documents:",
    len(eval_dataset),
)

print("Evaluation engine: READY")

Evaluation documents: 50
Evaluation engine: READY


In [22]:
## Diagnostic — Inspect Encoded Evaluation Sample Shapes

import numpy as np

sample = eval_dataset[0]

print("Evaluation dataset rows:", len(eval_dataset))
print("\nSample field shapes:")

for key, value in sample.items():
    try:
        print(f"{key:15s} -> {np.asarray(value).shape}")
    except Exception:
        print(f"{key:15s} -> {type(value)}")

print("\nExpected per-sample shapes:")
print("input_ids      -> (512,)")
print("attention_mask -> (512,)")
print("bbox           -> (512, 4)")
print("pixel_values   -> (3, 224, 224)")
print("labels         -> (512,)")

Evaluation dataset rows: 50

Sample field shapes:
input_ids       -> (512,)
attention_mask  -> (512,)
bbox            -> (512, 4)
labels          -> (512,)
pixel_values    -> (3, 224, 224)

Expected per-sample shapes:
input_ids      -> (512,)
attention_mask -> (512,)
bbox           -> (512, 4)
pixel_values   -> (3, 224, 224)
labels         -> (512,)


In [24]:
## 11. Generate Predictions from the Fine-Tuned Model

prediction_output = evaluator.predict(
    eval_dataset
)

logits = prediction_output.predictions
label_ids = prediction_output.label_ids

print("Logits shape:", logits.shape)
print("Labels shape:", label_ids.shape)

assert logits.shape[:2] == label_ids.shape

print("Baseline prediction generation: PASSED")

Logits shape: (50, 512, 7)
Labels shape: (50, 512)
Baseline prediction generation: PASSED


In [25]:
## 12. Decode Predictions and Calculate Confidence Scores

probabilities = torch.softmax(
    torch.tensor(logits),
    dim=-1,
).numpy()

prediction_ids = np.argmax(
    probabilities,
    axis=-1,
)

confidence_scores = np.max(
    probabilities,
    axis=-1,
)

print(
    "Prediction IDs:",
    prediction_ids.shape,
)

print(
    "Confidence scores:",
    confidence_scores.shape,
)

assert prediction_ids.shape == label_ids.shape
assert confidence_scores.shape == label_ids.shape

print("Prediction decoding: PASSED")

Prediction IDs: (50, 512)
Confidence scores: (50, 512)
Prediction decoding: PASSED


In [26]:
## 13. Remove Ignored and Padding Tokens from Evaluation

true_predictions = []
true_labels = []
true_confidences = []

for prediction, labels, confidences in zip(
    prediction_ids,
    label_ids,
    confidence_scores,
):
    document_predictions = []
    document_labels = []
    document_confidences = []

    for pred_id, label_id, confidence in zip(
        prediction,
        labels,
        confidences,
    ):
        if label_id == -100:
            continue

        document_predictions.append(
            label_list[int(pred_id)]
        )

        document_labels.append(
            label_list[int(label_id)]
        )

        document_confidences.append(
            float(confidence)
        )

    true_predictions.append(
        document_predictions
    )

    true_labels.append(
        document_labels
    )

    true_confidences.append(
        document_confidences
    )

print(
    "Documents evaluated:",
    len(true_labels),
)

print(
    "Valid tokens:",
    sum(
        len(document)
        for document in true_labels
    ),
)

assert len(true_labels) == 50

print("Evaluation token filtering: PASSED")

Documents evaluated: 50
Valid tokens: 8356
Evaluation token filtering: PASSED


In [27]:
## 14. Calculate Overall Baseline Performance

from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)

overall_metrics = {
    "precision": float(
        precision_score(
            true_labels,
            true_predictions,
        )
    ),

    "recall": float(
        recall_score(
            true_labels,
            true_predictions,
        )
    ),

    "f1": float(
        f1_score(
            true_labels,
            true_predictions,
        )
    ),

    "accuracy": float(
        accuracy_score(
            true_labels,
            true_predictions,
        )
    ),
}

for metric, value in overall_metrics.items():
    print(
        f"{metric:10s}: {value:.6f}"
    )

print("\nOverall baseline evaluation: COMPLETE")

precision : 0.741017
recall    : 0.804866
f1        : 0.771623
accuracy  : 0.797750

Overall baseline evaluation: COMPLETE


In [28]:
## 15. Verify Evaluation Reproducibility Against Saved Results

with Path(
    "results/baseline_metrics.json"
).open(
    "r",
    encoding="utf-8",
) as file:
    saved_baseline = json.load(file)

print(
    f"{'Metric':<12}"
    f"{'Training':>14}"
    f"{'Evaluation':>14}"
    f"{'Difference':>14}"
)

print("-" * 54)

for metric in [
    "precision",
    "recall",
    "f1",
    "accuracy",
]:
    previous = float(
        saved_baseline[metric]
    )

    reproduced = float(
        overall_metrics[metric]
    )

    difference = abs(
        previous - reproduced
    )

    print(
        f"{metric:<12}"
        f"{previous:>14.6f}"
        f"{reproduced:>14.6f}"
        f"{difference:>14.8f}"
    )

Metric            Training    Evaluation    Difference
------------------------------------------------------
precision         0.738440      0.741017    0.00257728
recall            0.801318      0.804866    0.00354790
f1                0.768595      0.771623    0.00302789
accuracy          0.792963      0.797750    0.00478698


In [29]:
## 16. Calculate Per-Class Entity Performance

from seqeval.metrics import (
    classification_report,
)

report = classification_report(
    true_labels,
    true_predictions,
    output_dict=True,
    zero_division=0,
)

entity_classes = [
    key
    for key, value in report.items()
    if isinstance(value, dict)
    and key not in [
        "micro avg",
        "macro avg",
        "weighted avg",
    ]
]

print("Detected entity classes:")

for entity_class in entity_classes:
    metrics = report[entity_class]

    print(
        f"\n{entity_class}"
    )

    print(
        "  Precision:",
        round(metrics["precision"], 4),
    )

    print(
        "  Recall:   ",
        round(metrics["recall"], 4),
    )

    print(
        "  F1:       ",
        round(metrics["f1-score"], 4),
    )

    print(
        "  Support:  ",
        int(metrics["support"]),
    )

Detected entity classes:

ANSWER
  Precision: 0.7464
  Recall:    0.8335
  F1:        0.7876
  Support:   805

HEADER
  Precision: 0.3932
  Recall:    0.3866
  F1:        0.3898
  Support:   119

QUESTION
  Precision: 0.7728
  Recall:    0.8303
  F1:        0.8006
  Support:   1049


In [30]:
## 17. Analyze Predicted and Ground-Truth Label Distributions

from collections import Counter

predicted_distribution = Counter(
    label
    for document in true_predictions
    for label in document
)

ground_truth_distribution = Counter(
    label
    for document in true_labels
    for label in document
)

print("GROUND TRUTH")
print("-" * 40)

for label, count in sorted(
    ground_truth_distribution.items()
):
    print(
        f"{label:<20} {count}"
    )

print("\nPREDICTIONS")
print("-" * 40)

for label, count in sorted(
    predicted_distribution.items()
):
    print(
        f"{label:<20} {count}"
    )

GROUND TRUTH
----------------------------------------
B-ANSWER             805
B-HEADER             119
B-QUESTION           1049
I-ANSWER             2474
I-HEADER             255
I-QUESTION           1423
O                    2231

PREDICTIONS
----------------------------------------
B-ANSWER             842
B-HEADER             92
B-QUESTION           1071
I-ANSWER             2150
I-HEADER             290
I-QUESTION           1324
O                    2587


In [31]:
## 18. Measure Prediction Confidence

all_confidences = np.array([
    confidence
    for document in true_confidences
    for confidence in document
])

confidence_statistics = {
    "mean": float(
        np.mean(all_confidences)
    ),

    "median": float(
        np.median(all_confidences)
    ),

    "std": float(
        np.std(all_confidences)
    ),

    "minimum": float(
        np.min(all_confidences)
    ),

    "maximum": float(
        np.max(all_confidences)
    ),

    "p10": float(
        np.percentile(
            all_confidences,
            10,
        )
    ),

    "p25": float(
        np.percentile(
            all_confidences,
            25,
        )
    ),

    "p75": float(
        np.percentile(
            all_confidences,
            75,
        )
    ),

    "p90": float(
        np.percentile(
            all_confidences,
            90,
        )
    ),
}

for key, value in confidence_statistics.items():
    print(
        f"{key:<10}: {value:.6f}"
    )

print(
    "\nTokens evaluated:",
    len(all_confidences),
)

mean      : 0.823073
median    : 0.882440
std       : 0.148580
minimum   : 0.263865
maximum   : 0.982482
p10       : 0.574221
p25       : 0.751408
p75       : 0.934225
p90       : 0.956538

Tokens evaluated: 8356


In [32]:
## 19. Compare Confidence for Correct and Incorrect Predictions

correct_confidence = []
incorrect_confidence = []

for predictions, labels, confidences in zip(
    true_predictions,
    true_labels,
    true_confidences,
):
    for prediction, label, confidence in zip(
        predictions,
        labels,
        confidences,
    ):
        if prediction == label:
            correct_confidence.append(
                confidence
            )
        else:
            incorrect_confidence.append(
                confidence
            )

print(
    "Correct predictions:",
    len(correct_confidence),
)

print(
    "Incorrect predictions:",
    len(incorrect_confidence),
)

print(
    "\nMean confidence — correct:",
    round(
        float(np.mean(correct_confidence)),
        6,
    ),
)

print(
    "Mean confidence — incorrect:",
    round(
        float(np.mean(incorrect_confidence)),
        6,
    ),
)

print(
    "\nToken error rate:",
    round(
        len(incorrect_confidence)
        /
        (
            len(correct_confidence)
            + len(incorrect_confidence)
        ),
        6,
    ),
)

Correct predictions: 6666
Incorrect predictions: 1690

Mean confidence — correct: 0.85519
Mean confidence — incorrect: 0.69639

Token error rate: 0.20225


In [33]:
## 20. Identify the Lowest-Confidence Model Decisions

low_confidence_records = []

for document_index, (
    predictions,
    labels,
    confidences,
) in enumerate(
    zip(
        true_predictions,
        true_labels,
        true_confidences,
    )
):
    for token_index, (
        prediction,
        label,
        confidence,
    ) in enumerate(
        zip(
            predictions,
            labels,
            confidences,
        )
    ):
        low_confidence_records.append({
            "document_index": document_index,
            "token_index": token_index,
            "true_label": label,
            "predicted_label": prediction,
            "confidence": float(confidence),
            "correct": prediction == label,
        })

low_confidence_records = sorted(
    low_confidence_records,
    key=lambda record: record["confidence"],
)

print("20 LOWEST-CONFIDENCE DECISIONS")
print("=" * 70)

for record in low_confidence_records[:20]:
    print(record)

20 LOWEST-CONFIDENCE DECISIONS
{'document_index': 30, 'token_index': 13, 'true_label': 'I-HEADER', 'predicted_label': 'B-ANSWER', 'confidence': 0.26386532187461853, 'correct': False}
{'document_index': 6, 'token_index': 104, 'true_label': 'I-QUESTION', 'predicted_label': 'O', 'confidence': 0.26785334944725037, 'correct': False}
{'document_index': 1, 'token_index': 106, 'true_label': 'O', 'predicted_label': 'I-ANSWER', 'confidence': 0.2925410568714142, 'correct': False}
{'document_index': 24, 'token_index': 67, 'true_label': 'B-QUESTION', 'predicted_label': 'I-QUESTION', 'confidence': 0.2933836877346039, 'correct': False}
{'document_index': 24, 'token_index': 23, 'true_label': 'O', 'predicted_label': 'O', 'confidence': 0.3058326542377472, 'correct': True}
{'document_index': 16, 'token_index': 8, 'true_label': 'B-HEADER', 'predicted_label': 'B-ANSWER', 'confidence': 0.310865581035614, 'correct': False}
{'document_index': 13, 'token_index': 55, 'true_label': 'I-ANSWER', 'predicted_label':

In [34]:
## 21. Identify the Most Common Label Confusions

confusion_counter = Counter()

for predictions, labels in zip(
    true_predictions,
    true_labels,
):
    for prediction, label in zip(
        predictions,
        labels,
    ):
        if prediction != label:
            confusion_counter[
                (label, prediction)
            ] += 1

print("MOST COMMON TOKEN-LEVEL CONFUSIONS")
print("=" * 70)

for (
    true_label,
    predicted_label,
), count in confusion_counter.most_common(20):

    print(
        f"{true_label:<20}"
        f" -> "
        f"{predicted_label:<20}"
        f"{count:>6}"
    )

MOST COMMON TOKEN-LEVEL CONFUSIONS
I-ANSWER             -> O                      413
I-QUESTION           -> O                      233
O                    -> I-ANSWER               123
O                    -> I-HEADER               111
O                    -> I-QUESTION              99
I-QUESTION           -> I-ANSWER                72
I-ANSWER             -> I-QUESTION              66
I-ANSWER             -> B-ANSWER                66
I-HEADER             -> O                       48
B-HEADER             -> B-QUESTION              47
I-HEADER             -> I-QUESTION              46
B-QUESTION           -> I-QUESTION              36
B-ANSWER             -> B-QUESTION              34
I-QUESTION           -> I-HEADER                31
B-QUESTION           -> O                       31
B-ANSWER             -> O                       29
O                    -> B-QUESTION              28
B-QUESTION           -> B-ANSWER                27
O                    -> B-ANSWER               

In [35]:
## 22. Measure Error Rate for Every Evaluation Document

document_error_records = []

for document_index, (
    predictions,
    labels,
) in enumerate(
    zip(
        true_predictions,
        true_labels,
    )
):
    total = len(labels)

    errors = sum(
        prediction != label
        for prediction, label in zip(
            predictions,
            labels,
        )
    )

    error_rate = (
        errors / total
        if total > 0
        else 0.0
    )

    document_error_records.append({
        "document_index": document_index,
        "evaluated_tokens": total,
        "errors": errors,
        "error_rate": float(error_rate),
    })

worst_documents = sorted(
    document_error_records,
    key=lambda record: record["error_rate"],
    reverse=True,
)

print("10 HIGHEST-ERROR DOCUMENTS")
print("=" * 70)

for record in worst_documents[:10]:
    print(record)

10 HIGHEST-ERROR DOCUMENTS
{'document_index': 0, 'evaluated_tokens': 223, 'errors': 142, 'error_rate': 0.6367713004484304}
{'document_index': 33, 'evaluated_tokens': 80, 'errors': 44, 'error_rate': 0.55}
{'document_index': 30, 'evaluated_tokens': 75, 'errors': 35, 'error_rate': 0.4666666666666667}
{'document_index': 45, 'evaluated_tokens': 240, 'errors': 102, 'error_rate': 0.425}
{'document_index': 1, 'evaluated_tokens': 167, 'errors': 70, 'error_rate': 0.41916167664670656}
{'document_index': 21, 'evaluated_tokens': 236, 'errors': 97, 'error_rate': 0.4110169491525424}
{'document_index': 15, 'evaluated_tokens': 204, 'errors': 79, 'error_rate': 0.3872549019607843}
{'document_index': 24, 'evaluated_tokens': 80, 'errors': 27, 'error_rate': 0.3375}
{'document_index': 3, 'evaluated_tokens': 222, 'errors': 74, 'error_rate': 0.3333333333333333}
{'document_index': 4, 'evaluated_tokens': 108, 'errors': 31, 'error_rate': 0.28703703703703703}


In [36]:
## 23. Save Reproducible Prediction Artifacts for Error Analysis

import pickle

ARTIFACT_DIR = Path(
    "/content/docshield-evaluation-artifacts"
)

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

artifact = {
    "true_predictions": true_predictions,
    "true_labels": true_labels,
    "true_confidences": true_confidences,
    "document_error_records": document_error_records,
    "overall_metrics": overall_metrics,
    "per_class_report": report,
    "confidence_statistics": confidence_statistics,
}

ARTIFACT_PATH = (
    ARTIFACT_DIR
    / "baseline_predictions.pkl"
)

with ARTIFACT_PATH.open("wb") as file:
    pickle.dump(
        artifact,
        file,
    )

print(
    "Artifact:",
    ARTIFACT_PATH,
)

print(
    "Exists:",
    ARTIFACT_PATH.exists(),
)

print(
    "Size:",
    ARTIFACT_PATH.stat().st_size,
    "bytes",
)

assert ARTIFACT_PATH.exists()

print("Evaluation artifact: SAVED")

Artifact: /content/docshield-evaluation-artifacts/baseline_predictions.pkl
Exists: True
Size: 111838 bytes
Evaluation artifact: SAVED


In [37]:
## 24. Summarize the Baseline Evaluation Study

print("=" * 72)
print("DOCSHIELD — BASELINE EVALUATION")
print("=" * 72)

print(
    f"Documents evaluated : "
    f"{len(true_labels)}"
)

print(
    f"Tokens evaluated    : "
    f"{sum(len(x) for x in true_labels)}"
)

print(
    f"Precision           : "
    f"{overall_metrics['precision']:.4f}"
)

print(
    f"Recall              : "
    f"{overall_metrics['recall']:.4f}"
)

print(
    f"F1                  : "
    f"{overall_metrics['f1']:.4f}"
)

print(
    f"Accuracy            : "
    f"{overall_metrics['accuracy']:.4f}"
)

print(
    f"Mean confidence     : "
    f"{confidence_statistics['mean']:.4f}"
)

print(
    f"Entity classes      : "
    f"{', '.join(entity_classes)}"
)

print("=" * 72)
print("BASELINE EVALUATION: COMPLETE")
print("=" * 72)

DOCSHIELD — BASELINE EVALUATION
Documents evaluated : 50
Tokens evaluated    : 8356
Precision           : 0.7410
Recall              : 0.8049
F1                  : 0.7716
Accuracy            : 0.7978
Mean confidence     : 0.8231
Entity classes      : ANSWER, HEADER, QUESTION
BASELINE EVALUATION: COMPLETE
